# Практическая работа. Кластерный анализ. K-Means

Цель работы: посмотреть, как работает K-Means на простом датасете Iris и на реальных данных по регионам России. Писал ответы простыми словами, как обычный отчет по практике.

## Задание 1. Iris

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target

df_iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [2]:
print('shape:', df_iris.shape)
display(df_iris.describe())
print('Пропуски:')
print(df_iris.isna().sum())

shape: (150, 5)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232
min,4.300000,2.000000,1.000000,0.100000,0.000000
25%,5.100000,2.800000,1.600000,0.300000,0.000000
50%,5.800000,3.000000,4.350000,1.300000,1.000000
75%,6.400000,3.300000,5.100000,1.800000,2.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000


Пропуски:
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
target               0
dtype: int64


В датасете Iris 4 числовых признака: длина и ширина чашелистика, длина и ширина лепестка. Пропусков нет, потому что по всем столбцам количество NaN равно нулю. Столбец target — это настоящий класс цветка, то есть вид ириса. Он нужен не для обучения K-Means, а чтобы потом примерно сравнить, насколько кластеры похожи на реальные виды.

## Подготовка данных Iris

In [3]:
X_iris = df_iris[iris.feature_names]
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

print(X_iris_scaled[:5])

[[-0.90068117  1.01900435 -1.34022653 -1.3154443 ]
 [-1.14301691 -0.13197948 -1.34022653 -1.3154443 ]
 [-1.38535265  0.32841405 -1.39706395 -1.3154443 ]
 [-1.50652052  0.09821729 -1.2833891  -1.3154443 ]
 [-1.02184904  1.24920112 -1.34022653 -1.3154443 ]]


Масштабирование важно, потому что K-Means считает расстояния между точками. Если один признак будет в больших числах, он начнет сильнее влиять на расстояние, даже если по смыслу он не важнее остальных. Поэтому признаки приводят примерно к одному масштабу.

## K-Means для Iris

In [4]:
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=1)
kmeans_iris.fit(X_iris_scaled)
labels_iris = kmeans_iris.labels_
centers_iris_scaled = kmeans_iris.cluster_centers_
centers_iris = scaler_iris.inverse_transform(centers_iris_scaled)

df_iris['cluster'] = labels_iris

centers_df = pd.DataFrame(centers_iris, columns=iris.feature_names)
centers_df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,6.314583,2.895833,4.973958,1.703125
1,5.169697,3.630303,1.493939,0.272727
2,4.747619,2.895238,1.757143,0.352381


Центроид — это как средняя точка кластера. Например, если смотреть на координату центроида по длине лепестка, то она показывает примерную среднюю длину лепестка у цветов, которые попали в этот кластер. То есть по центроидам можно понять, какие кластеры состоят из маленьких, средних или крупных цветков.

## Графики Iris

In [5]:
plt.figure(figsize=(7,5))
plt.scatter(df_iris['petal length (cm)'], df_iris['petal width (cm)'], c=df_iris['cluster'])
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.title('Iris: кластеры K-Means')
plt.show()

/tmp/ipykernel_8604/900304991.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
plt.figure(figsize=(7,5))
plt.scatter(df_iris['petal length (cm)'], df_iris['petal width (cm)'], c=df_iris['target'])
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.title('Iris: настоящие виды')
plt.show()

/tmp/ipykernel_8604/3909258530.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


По графикам видно, что один вид ирисов отделяется очень хорошо. Это цветы с маленькими лепестками. Два других вида разделяются хуже, потому что часть точек находится близко друг к другу и немного смешивается. В целом K-Means неплохо поймал структуру, но идеально реальные виды он не повторил.

## Метод локтя для Iris

In [7]:
wcss_iris = []
ks_iris = range(1, 11)
for k in ks_iris:
    model = KMeans(n_clusters=k, random_state=42, n_init=1)
    model.fit(X_iris_scaled)
    wcss_iris.append(model.inertia_)

plt.figure(figsize=(7,5))
plt.plot(list(ks_iris), wcss_iris, marker='o')
plt.xlabel('k')
plt.ylabel('WCSS / inertia')
plt.title('Метод локтя для Iris')
plt.grid(True)
plt.show()

pd.DataFrame({'k': list(ks_iris), 'inertia': wcss_iris})

/tmp/ipykernel_8604/3930628980.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,k,inertia
0,1,600.000000
1,2,222.361705
2,3,191.024737
3,4,114.354072
4,5,91.047670
5,6,81.550757
6,7,80.777739
7,8,64.426748
8,9,55.706900
9,10,51.121095


По методу локтя для Iris разумно взять k = 3. До этого значения ошибка падает довольно заметно, а дальше улучшение уже становится более плавным. Это еще нормально совпадает с тем, что в Iris реально есть 3 вида цветков.

## Сравнение кластеров с настоящими видами

In [8]:
pd.crosstab(df_iris['target'], df_iris['cluster'])

cluster,0,1,2
target,,,
0,0,33,17
1,46,0,4
2,50,0,0


По таблице видно, что один настоящий вид почти полностью попал в отдельный кластер. Остальные два вида разделились не так идеально, между ними есть пересечение. Для K-Means это нормальная ситуация, потому что алгоритм не знает настоящие классы и просто делит данные по расстояниям.

Вывод по Iris: датасет удобный для первого знакомства с кластеризацией, потому что в нем нет пропусков и все признаки числовые. K-Means смог хорошо отделить один вид ириса и частично разделил два других. Значит, структура в данных есть, но она не совсем идеально кластерная.

## Задание 2. Данные Росстата по зерну

In [9]:
raw_path = 'rus_grain_regions_simple.csv'
df_raw = pd.read_csv(raw_path, sep=';', encoding='utf-8', header=None, dtype=str)

print('shape:', df_raw.shape)
display(df_raw.head(12))
print(df_raw.dtypes)

shape: (95, 4)


,0,1,2,3
0,2025 год,NaN,NaN,NaN
1,Хозяйства всех категорий,NaN,NaN,NaN
2,Зерновые и зернобобовые культуры,NaN,NaN,NaN
3,Регион,Валовой сбор (тыс. центнеров),Урожайность (ц с 1 га убранной площади),Посевные площади (тыс. гектаров)
4,Российская Федерация,"1.411.523,4","33,1","43.663,4"
5,Центральный федеральный округ,"357.983,5","48,4","7.632,6"
6,Белгородская область,"30.996,6","54,9","582,5"
7,Брянская область,"17.134,8","62,2","292,8"
8,Владимирская область,"3.019,9","33,2","92,8"
9,Воронежская область,"61.298,5","42,3","1.480,5"


0    object
1    object
2    object
3    object
dtype: object


В исходной таблице есть не только субъекты РФ, но и служебные строки сверху, строка по всей Российской Федерации и строки по федеральным округам. Для кластеризации субъектов РФ их надо убрать, иначе в одной таблице смешаются разные уровни данных.

## Предобработка данных Росстата

In [10]:
df = pd.read_csv(raw_path, sep=';', encoding='utf-8', skiprows=3, dtype=str)
df.columns = ['region', 'gross_harvest_thousand_centners', 'yield_centners_per_ha', 'sown_area_thousand_ha']

# убираем пустые строки и лишние пробелы
df = df.dropna(how='all').copy()
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

# убираем РФ в целом и федеральные округа
mask_total = df['region'].eq('Российская Федерация')
mask_district = df['region'].str.contains('федеральный округ', case=False, na=False)
df = df[~mask_total & ~mask_district].copy()

# убираем специальную дубль-строку по Тюменской области, чтобы не считать один и тот же регион два раза
df = df[~df['region'].str.contains('Тюменская область \(кроме', regex=True, na=False)].copy()

def to_float_ru(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace('"', '')
    if x in ['', '...', 'nan']:
        return np.nan
    x = x.replace('.', '').replace(',', '.')
    return pd.to_numeric(x, errors='coerce')

num_cols = ['gross_harvest_thousand_centners', 'yield_centners_per_ha', 'sown_area_thousand_ha']
for col in num_cols:
    df[col] = df[col].apply(to_float_ru)

excluded_missing = df[df[num_cols].isna().any(axis=1)][['region'] + num_cols]
print('Исключены из-за пропусков:')
display(excluded_missing)

df_clean = df.dropna(subset=num_cols).copy()
df_clean.to_csv('rus_grain_regions_clean.csv', index=False, encoding='utf-8-sig')

print('Размер очищенного датасета:', df_clean.shape)
display(df_clean.head())

Исключены из-за пропусков:


<>:15: SyntaxWarning: invalid escape sequence '\('
<>:15: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_8604/3570991325.py:15: SyntaxWarning: invalid escape sequence '\('
  df = df[~df['region'].str.contains('Тюменская область \(кроме', regex=True, na=False)].copy()


,region,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha
8,Костромская область,NaN,21.5,19.9
19,г. Москва,NaN,33.1,0.9
22,Республика Коми,NaN,9.9,0.0
23,Архангельская область,NaN,21.8,0.5
24,Архангельская область (кроме Ненецкого автоном...,NaN,21.8,0.5
35,Астраханская область,NaN,37.0,23.2
38,г. Севастополь,NaN,47.6,0.2


Размер очищенного датасета: (71, 4)


,region,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha
2,Белгородская область,30996.6,54.9,582.5
3,Брянская область,17134.8,62.2,292.8
4,Владимирская область,3019.9,33.2,92.8
5,Воронежская область,61298.5,42.3,1480.5
6,Ивановская область,1445.2,26.9,55.0


Я убрал служебные строки, строку по России целиком и федеральные округа, потому что они не являются обычными субъектами РФ. Еще убрал строку “Тюменская область (кроме ...)”, так как рядом уже есть “Тюменская область” с такими же числами, и это дало бы дубль.

Регионы с `...` в числах я исключил, потому что валовой сбор не получилось нормально перевести в число. Оставлять такие строки в K-Means нельзя: алгоритму нужны числовые признаки без пропусков.

## Первичный анализ очищенных данных

In [11]:
print('shape:', df_clean.shape)
display(df_clean.describe())

for col in num_cols:
    print('\n', col)
    print('min:', df_clean.loc[df_clean[col].idxmin(), ['region', col]].to_dict())
    print('max:', df_clean.loc[df_clean[col].idxmax(), ['region', col]].to_dict())

shape: (71, 4)


,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha
count,71.000000,71.000000,71.000000
mean,19862.652113,33.076056,614.346479
std,24994.987020,14.143009,784.240148
min,0.300000,7.200000,0.000000
25%,2311.450000,23.250000,81.300000
50%,8137.700000,29.700000,286.100000
75%,31028.300000,40.300000,785.400000
max,116389.200000,66.600000,3547.600000



 gross_harvest_thousand_centners
min: {'region': 'Республика Карелия', 'gross_harvest_thousand_centners': 0.3}
max: {'region': 'Краснодарский край', 'gross_harvest_thousand_centners': 116389.2}

 yield_centners_per_ha
min: {'region': 'Республика Тыва', 'yield_centners_per_ha': 7.2}
max: {'region': 'Приморский край', 'yield_centners_per_ha': 66.6}

 sown_area_thousand_ha
min: {'region': 'Республика Карелия', 'sown_area_thousand_ha': 0.0}
max: {'region': 'Ростовская область', 'sown_area_thousand_ha': 3547.6}


По данным видно, что регионы очень разные по масштабу. Есть регионы с огромными площадями и валовым сбором, а есть регионы, где зерновое производство совсем небольшое. По урожайности тоже большой разброс: где-то она высокая, а где-то заметно ниже, что может быть связано с климатом и условиями земледелия.

## Подготовка признаков для K-Means

In [12]:
features = ['gross_harvest_thousand_centners', 'yield_centners_per_ha', 'sown_area_thousand_ha']
X = df_clean[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled[:5])

[[ 0.44861772  1.55407355 -0.0408971 ]
 [-0.10991275  2.07390337 -0.41292848]
 [-0.67864132  0.00882597 -0.66976755]
 [ 1.66956553  0.65683302  1.1123103 ]
 [-0.74209036 -0.43979429 -0.71831013]]


Масштабирование здесь особенно важно. Валовой сбор измеряется тысячами центнеров и может быть очень большим числом, а урожайность обычно намного меньше по значениям. Если не масштабировать данные, K-Means почти полностью будет ориентироваться на самые большие по масштабу признаки.

## Кластеризация регионов при k = 3

In [13]:
kmeans_regions = KMeans(n_clusters=3, random_state=42, n_init=1)
df_clean['cluster'] = kmeans_regions.fit_predict(X_scaled)

centers_regions = pd.DataFrame(
    scaler.inverse_transform(kmeans_regions.cluster_centers_),
    columns=features
)
centers_regions.index.name = 'cluster'
display(centers_regions)

display(df_clean[['region'] + features + ['cluster']].sort_values('cluster').head(20))

,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha
cluster,,,
0,64906.490909,30.927273,2179.154545
1,7633.408889,26.264444,280.297778
2,23518.233333,55.086667,468.966667


,region,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha,cluster
5,Воронежская область,61298.5,42.3,1480.5,0
34,Краснодарский край,116389.2,48.4,2457.6,0
37,Ростовская область,89222.3,26.1,3547.6,0
36,Волгоградская область,58661.5,27.6,2158.0,0
60,Саратовская область,55609.2,25.4,2244.1,0
51,Республика Татарстан (Татарстан),46280.1,35.9,1290.4,0
46,Ставропольский край,105992.3,43.8,2434.1,0
48,Республика Башкортостан,36387.8,26.7,1381.1,0
77,Омская область,40008.3,23.1,1880.4,0
72,Алтайский край,62555.0,23.0,2738.9,0


In [14]:
plt.figure(figsize=(8,5))
plt.scatter(df_clean['sown_area_thousand_ha'], df_clean['yield_centners_per_ha'], c=df_clean['cluster'])
plt.xlabel('Посевные площади, тыс. га')
plt.ylabel('Урожайность, ц/га')
plt.title('Кластеры регионов: площадь и урожайность')
plt.grid(True)
plt.show()

/tmp/ipykernel_8604/847581953.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Выбор числа кластеров

In [15]:
rows = []
for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=1)
    labels = model.fit_predict(X_scaled)
    rows.append({
        'k': k,
        'inertia': model.inertia_,
        'silhouette': silhouette_score(X_scaled, labels)
    })

metrics = pd.DataFrame(rows)
display(metrics)

plt.figure(figsize=(7,5))
plt.plot(metrics['k'], metrics['inertia'], marker='o')
plt.xlabel('k')
plt.ylabel('inertia')
plt.title('Метод локтя для регионов')
plt.grid(True)
plt.show()

plt.figure(figsize=(7,5))
plt.plot(metrics['k'], metrics['silhouette'], marker='o')
plt.xlabel('k')
plt.ylabel('silhouette_score')
plt.title('Метод силуэта для регионов')
plt.grid(True)
plt.show()

,k,inertia,silhouette
0,2,117.155864,0.483797
1,3,64.595933,0.512989
2,4,52.568558,0.357465
3,5,41.418525,0.369957
4,6,30.742373,0.390871
5,7,22.259226,0.418033
6,8,19.308513,0.398824
7,9,18.044780,0.361239
8,10,15.769297,0.343722


/tmp/ipykernel_8604/977879450.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_8604/977879450.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


По графикам я бы оставил k = 3. Это не значит, что это единственно правильный вариант, но для отчета он нормально читается: можно разделить регионы на крупные производящие, средние/обычные и небольшие или слабые по показателям. Если брать больше кластеров, детализация растет, но интерпретировать становится сложнее.

## Интерпретация кластеров

In [16]:
cluster_summary = df_clean.groupby('cluster')[features].mean().round(2)
cluster_counts = df_clean['cluster'].value_counts().sort_index().rename('count')
display(pd.concat([cluster_counts, cluster_summary], axis=1))

for cl in sorted(df_clean['cluster'].unique()):
    print(f'Кластер {cl}:')
    print(', '.join(df_clean.loc[df_clean['cluster'] == cl, 'region'].tolist()))
    print()

,count,gross_harvest_thousand_centners,yield_centners_per_ha,sown_area_thousand_ha
cluster,,,,
0,11,64906.49,30.93,2179.15
1,45,7633.41,26.26,280.30
2,15,23518.23,55.09,468.97


Кластер 0:
Воронежская область, Краснодарский край, Волгоградская область, Ростовская область, Ставропольский край, Республика Башкортостан, Республика Татарстан (Татарстан), Оренбургская область, Саратовская область, Алтайский край, Омская область

Кластер 1:
Владимирская область, Ивановская область, Калужская область, Московская область, Смоленская область, Тверская область, Ярославская область, Республика Карелия, Вологодская область, Ленинградская область, Новгородская область, Псковская область, Республика Калмыкия, Республика Крым, Республика Дагестан, Республика Ингушетия, Республика Северная Осетия-Алания, Чеченская Республика, Республика Марий Эл, Республика Мордовия, Удмуртская Республика, Чувашская Республика - Чувашия, Пермский край, Кировская область, Нижегородская область, Самарская область, Ульяновская область, Курганская область, Свердловская область, Тюменская область, Челябинская область, Республика Алтай, Республика Тыва, Республика Хакасия, Красноярский край, Иркутс

Кластеры получились примерно такие. Один кластер — это регионы с очень большим масштабом производства: там большие посевные площади и большой валовой сбор. Другой кластер — регионы со средними или небольшими площадями, но иногда с хорошей урожайностью. Еще один кластер — регионы, где производство зерна небольшое или условия менее подходящие, поэтому показатели ниже.

В целом географическая структура просматривается: крупные зерновые регионы чаще связаны с территориями, где земледелие развито сильнее и площади больше. Но K-Means смотрит только на числа, а не на карту, поэтому он не всегда будет группировать соседние регионы вместе.

## Ограничения K-Means

У K-Means есть несколько ограничений. Во-первых, число кластеров k надо задавать заранее, а в реальных данных не всегда понятно, сколько групп реально есть. Во-вторых, метод чувствителен к масштабу признаков, поэтому без StandardScaler результат может быть перекошен. В-третьих, выбросы могут сильно сдвигать центроиды. Еще K-Means лучше работает, когда кластеры похожи на круглые группы вокруг центров, а реальные региональные данные могут быть сложнее.

В аграрной статистике сложная форма кластеров может появляться, если регионы отличаются не по одному направлению, а сразу по нескольким: климат, площадь, специализация, технологии, доля сельхозземель и т.д. Тогда регионы могут образовывать вытянутые или смешанные группы, которые K-Means описывает не очень точно.

## Контрольные вопросы

1. Кластерный анализ — это способ разделить объекты на группы без заранее известных ответов. Он нужен, чтобы найти похожие объекты и увидеть структуру в данных.

2. Идея K-Means в том, что алгоритм выбирает центры кластеров, относит точки к ближайшему центру, потом пересчитывает центры и повторяет это несколько раз. В итоге точки должны оказаться около своих центроидов.

3. Масштабировать признаки важно, потому что K-Means использует расстояния. Если один признак измеряется большими числами, он будет влиять на результат сильнее остальных.

4. Коэффициент силуэта показывает, насколько объект похож на свой кластер и насколько он далек от других кластеров. Чем значение выше, тем обычно лучше разделение.

5. Предобработка нужна, потому что реальные данные часто содержат пропуски, лишние строки, разные форматы чисел и дубли. Если это не исправить, модель может либо не запуститься, либо дать неправильный результат.

6. Субъекты РФ и федеральные округа нельзя смешивать, потому что это разные уровни данных. Округ включает несколько регионов, поэтому его показатели намного крупнее и будут ломать сравнение.

7. Ограничения K-Means при анализе региональной статистики такие: надо заранее выбрать k, метод чувствителен к масштабу и выбросам, а еще он плохо работает со сложными формами кластеров.

## Общий вывод

В этой работе я применил K-Means сначала к Iris, а потом к данным по регионам России. На Iris алгоритм хорошо отделил один вид цветков и частично разделил два других. На данных Росстата пришлось сначала очистить таблицу, убрать лишние уровни данных и привести числа к нормальному формату. После масштабирования K-Means разделил регионы на группы по урожайности, посевным площадям и валовому сбору. В целом метод полезен для первичного анализа, но его результат надо интерпретировать аккуратно, потому что он зависит от выбранного k, масштаба признаков и выбросов.